In [7]:
import pandas as pd
import numpy as np
import os

# Define environment paths
RAW_DIR = "../data/raw/"
PROCESSED_DIR = "../data/processed/"

print("--- 1. TABULAR FEATURE ENGINEERING (RFM) ---")

# 1. Load the data sources we actually need
df_erp = pd.read_parquet(os.path.join(RAW_DIR, "erp_transactions.parquet"))
df_support = pd.read_parquet(os.path.join(RAW_DIR, "supporting_documents.parquet"))

# 2. Extract Recency (R) and Frequency (F) from the ERP headers
print("Calculating Recency and Frequency from ERP logs...")
df_erp['CREATIONDATE'] = pd.to_datetime(df_erp['CREATIONDATE'])
max_dataset_date = df_erp['CREATIONDATE'].max()

rf_features = df_erp.groupby('SOLDTOPARTY').agg(
    total_orders=('SALESDOCUMENT', 'nunique'),          # F: Frequency
    last_order_date=('CREATIONDATE', 'max')             # R: Recency base
).reset_index()

rf_features['days_since_last_order'] = (max_dataset_date - rf_features['last_order_date']).dt.days
rf_features.rename(columns={'SOLDTOPARTY': 'customer_id'}, inplace=True)
rf_features['customer_id'] = rf_features['customer_id'].astype(str)

# 3. Extract Monetary (M) value from the Supporting Documents (Purchase Orders)
print("Extracting Monetary values from Purchase Orders...")
# Filter only for purchase orders to avoid double-counting invoices
purchase_orders = df_support[df_support['document_type'] == 'PURCHASE_ORDER'].copy()

# Ensure financial columns are numeric
purchase_orders['total_amount'] = pd.to_numeric(purchase_orders['total_amount'], errors='coerce').fillna(0)
purchase_orders['quantity'] = pd.to_numeric(purchase_orders['quantity'], errors='coerce').fillna(0)

m_features = purchase_orders.groupby('customer_id').agg(
    total_spend=('total_amount', 'sum'),                # M: Monetary
    avg_order_value=('total_amount', 'mean'),
    total_items_purchased=('quantity', 'sum')
).reset_index()
m_features['customer_id'] = m_features['customer_id'].astype(str)

# 4. Combine R, F, and M into one tabular dataset
tabular_features = pd.merge(rf_features, m_features, on='customer_id', how='left')

# Fill zeros for any customers that have ERP records but no matching PDF Purchase Orders yet
tabular_features = tabular_features.fillna({
    'total_spend': 0, 
    'avg_order_value': 0, 
    'total_items_purchased': 0
})

print(f"Generated tabular features for {len(tabular_features)} unique customers.")

print("\n--- 2. THE MASTER JOIN ---")

# Load the NLP features you generated previously
nlp_features = pd.read_parquet(os.path.join(PROCESSED_DIR, "nlp_customer_features.parquet"))
nlp_features['customer_id'] = nlp_features['customer_id'].astype(str)

# Merge the hard math (Tabular) with the human emotion (NLP)
master_df = pd.merge(tabular_features, nlp_features, on='customer_id', how='inner')

print(f"Successfully joined! Final multimodal dataset shape: {master_df.shape}")

print("\n--- 3. DEFINING THE TARGET VARIABLE ---")

# Define churn threshold: A customer who has not ordered anything in the last 60 days
CHURN_THRESHOLD_DAYS = 60
master_df['is_churned'] = (master_df['days_since_last_order'] > CHURN_THRESHOLD_DAYS).astype(int)

# Safely print the class balance
churn_dist = master_df['is_churned'].value_counts(normalize=True) * 100
print(f"Active Customers (0): {churn_dist.get(0, 0.0):.1f}%")
print(f"Churned Customers (1): {churn_dist.get(1, 0.0):.1f}%")

# Save the final training set
output_path = os.path.join(PROCESSED_DIR, "master_training_set.parquet")
master_df.drop(columns=['last_order_date']).to_parquet(output_path, index=False)

print(f"\nReady for model training. Saved to: {output_path}")

--- 1. TABULAR FEATURE ENGINEERING (RFM) ---
Calculating Recency and Frequency from ERP logs...
Extracting Monetary values from Purchase Orders...
Generated tabular features for 13155 unique customers.

--- 2. THE MASTER JOIN ---
Successfully joined! Final multimodal dataset shape: (13155, 11)

--- 3. DEFINING THE TARGET VARIABLE ---
Active Customers (0): 31.1%
Churned Customers (1): 68.9%

Ready for model training. Saved to: ../data/processed/master_training_set.parquet


In [8]:
# df_erp.columns